# Epsilon Fund - Walk-Forward Validation
Uses `infrastructure/walk_forward/` to run rolling Optuna optimisation and evaluate OOS robustness. Ideal to use after finding strategy that seems to work using backtesting framework to ensure logic is valid.

---
### Iteration Guidelines

**Overfitting the iteration process:** Each time you inspect OOS results and adjust parameters, you leak OOS information into your design decisions. Cap yourself at **3–4 iterations** — first run with everything free, second with obvious fixes from CV + plateau analysis, third to tighten remaining params. 

If the strategy still shows heavy overfitting signals after three passes, **the problem is the strategy architecture, not the parameters**.

**WFE:** Walk-forward efficiency - examine IS/OOS ratio (simplest).

**Pertubation degradation:** Examine pertubation table to see if degradation reduces across runs.


| Signal | Meaning | Action |
|--------|---------|--------|
| IS Sharpe drops, OOS Sharpe holds or rises, WFE improves | Removing noise-fitting degrees of freedom | Continue iterating |
| Perturbation degradation shrinks across iterations | Parameters becoming more robust | Continue iterating |
| N/A plateau params decreasing across iterations | Strategy becoming more tolerant of parameter movement | Continue iterating |
| WFE improvement flattens (e.g. 0.55 → 0.65 → 0.67) | Diminishing returns — further fixes won't help much | Stop iterating |
| IS and OOS both decline but WFE rises (IS falls faster) | Constraining away real signal, not just noise | Stop iterating |
| OOS Sharpe keeps declining despite "better" param setup | Overfitting the iteration process itself | Stop — problem is strategy architecture, not parameters |
| WFE decreases after fixing a parameter | Locked in a param that was legitimately adapting across folds | Unfix that parameter and re-run |

---

In [1]:
import sys
import os
import pandas as pd
import numpy as np


# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))


from binance_client import get_binance_client, get_data
from wf_engine import walk_forward, plateau_analysis, plateau_summary, perturbation_test, cost_stress_test
from wf_visualizer import plot_walk_forward_results, plot_plateau_analysis
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: must be >= (train_bars + test_bars) * n_folds desired

In [2]:
SYMBOL   = 'AVAXUSDT'
INTERVAL = '1d'
LOOKBACK = 2150 


# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client   = get_binance_client()
df = get_data(client, SYMBOL, INTERVAL, LOOKBACK)
print(f'Data: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
df.tail(3)

Data: 2020-09-22 → 2026-04-08  (2025 bars)


,Open,High,Low,Close,Volume
Time,,,,,
2026-04-06,9.07,9.66,8.81,8.91,3843829.97
2026-04-07,8.90,9.62,8.46,9.43,6882469.21
2026-04-08,9.43,9.56,9.32,9.35,1756740.99


In [3]:
# ── parameter search space ────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
# Only keys present in PARAMS above are searched — remove a key from PARAMS to exclude it entirely.

PARAM_DEFS = {
    'ema_span':          ('int',   14,    35),
    'swing_caution':     ('int',   3,    14),
    'swing_stop':        ('int',   3,    10),
    'atr_caution':       ('int',   10,   30),
    'atr_stop':          ('int',   10,   30),
    'atr_size':          ('int',   3,    14),
    'adx_override':      ('int',   40,   80),
  #  'adx_override':      ('int',   40,   70),

    'stop_atr_scale':    ('float', 0.5,  2.0),
    'risk_per_trade':    ('float', 0.005, 0.05),
    'max_leverage':      ('float', 1.0,  3.0),
    'stop_mult_pos_caution': ('float', 0.1, 0.9),
    'stop_mult_pos_normal':  ('float', 0.8, 2.0),
    'stop_mult_ent_both':    ('float', 0.5, 2.5),
    'stop_mult_ent_caution': ('float', 0.1, 0.9),
    'stop_mult_ent_short':   ('float', 0.1, 1.5),
    'stop_mult_ent_normal':  ('float', 0.5, 1.5),
    'vol_ma_period':  ('int',   10,  40),   # rolling window for volume MA
    'obv_ma_period':  ('int',   10,  40),   # OBV smoothing window
    'obv_lookback':   ('int',   10,  30),   # bars back to compare price vs OBV direction
}

# ── anchored params (set after a stability run; empty first time) ─────────────
# These bypass Optuna and are held constant across all folds.
# Populate using stability_df results: fix params with CV < 0.15
FIXED_PARAMS = { 
    'risk_per_trade': 0.0468,
    'max_leverage': 2.5,

    'stop_mult_pos_caution': 0.8423,
    'stop_atr_scale':1.9,

    #'stop_mult_ent_normal': 1,
   
    'vol_ma_period': 31,

    }

keep in mind: sme

### *Guide to parameter anchoring*

|  | Robust Plateau| Fragile Plateau |
|--|-------------------|-------------------|
| Low CV | Stable across folds and insensitive to exact value - keep free| Looks stable but is fitting the same noise patterns - fix at concensus|
| High CV | Parameter unimportant - fix at any reasonable value | Unstable across folds and sitting on a cliff - strong candidate to eliminate |

Copy-paste plateau analysis table above into fixed params section and decide manually on which to fix/keep free.c

---
## Strategy

**CRUCIAL** - Strategy logic needs to work well in backtesting notebook before running here, saves time not running walk-forward for a broken strategy.

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> The engine applies a 1-bar execution lag automatically. Inside the strategy loop, use `prev` for signal decisions and `curr` for execution — no manual shifting needed.

**To implement your strategy:**
1. Write strategy logic — compute indicators, signals, and execution loop: use `param_name`for those to be searched
2. Update `indicator_cols` to list your longest-warmup indicators — the engine uses this to clean NaN rows after OOS trimming


In [4]:
def my_strategy(df_slice: pd.DataFrame, params: dict) -> pd.DataFrame:

    df = df_slice.copy()

    # ── strategy logic ────────────────────────────────────────────────────────
    df['EMA']          = df['Close'].ewm(span=params['ema_span'], adjust=False).mean()
    df['Swing_Hi_Cau'] = df['High'].rolling(params['swing_caution']).max()
    df['Swing_Lo_Cau'] = df['Low'].rolling(params['swing_caution']).min()
    df['Swing_Hi_Stp'] = df['High'].rolling(params['swing_stop']).max()

    def atr(period):
        hl = df['High'] - df['Low']
        hc = (df['High'] - df['Close'].shift(1)).abs()
        lc = (df['Low']  - df['Close'].shift(1)).abs()
        return pd.concat([hl, hc, lc], axis=1).max(axis=1).ewm(span=period, adjust=False).mean()

    df['ATR_Cau'] = atr(params['atr_caution'])
    df['ATR_Stp'] = atr(params['atr_stop'])
    df['ATR_Sz']  = atr(params['atr_size'])

    up    = df['High'].diff();  down = -df['Low'].diff()
    pdm   = up.where((up > down) & (up > 0), 0.0)
    ndm   = down.where((down > up) & (down > 0), 0.0)
    atr14 = atr(14)
    pdi   = 100 * pdm.ewm(span=14, adjust=False).mean() / atr14
    ndi   = 100 * ndm.ewm(span=14, adjust=False).mean() / atr14
    dx    = (100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, np.nan)).fillna(0)
    df['ADX_14'] = dx.ewm(span=14, adjust=False).mean()

    df['Vol_MA']  = df['Volume'].rolling(params['vol_ma_period']).mean()
    direction     = df['Close'].diff().apply(lambda x: 1 if x > 0 else -1)
    df['OBV']     = (df['Volume'] * direction).cumsum()
    df['OBV_MA']  = df['OBV'].rolling(params['obv_ma_period']).mean()


    df['Caution_OBV']   = (df['Close'] > df['Close'].shift(params['obv_lookback'])) & (df['OBV'] < df['OBV_MA'])
    df['Caution_Long']  = ((df['Swing_Hi_Cau'] - df['Low']) > 1.5 * df['ATR_Cau']) | df['Caution_OBV']
    df['Caution_Short'] = ((df['High'] - df['Swing_Lo_Cau']) > 1.5 * df['ATR_Cau']) | (df['Close'] > df['EMA'])
    df['Entry_Long'] = (df['Close'] > df['EMA']) & (~df['Caution_Long'] | (df['ADX_14'] > params['adx_override'])) & (df['Volume'] > df['Vol_MA'])
    df['position_size_raw'] = (params['risk_per_trade'] / (df['ATR_Sz'] / df['Close'])).clip(0.1, params['max_leverage'])

    n             = len(df)
    position      = [0]     * n
    position_size = [0.0]   * n
    stop_arr      = [np.nan] * n
    in_position   = 0
    stop_loss     = np.nan
    current_size  = 0.0

    for i in range(1, n):
        prev = df.iloc[i - 1]
        curr = df.iloc[i]

        if in_position == 0:
            if prev['Entry_Long']:
                in_position  = 1
                current_size = curr['position_size_raw']
                cl = prev['Caution_Long']; cs = prev['Caution_Short']
                if cl and cs: sm = params['stop_mult_ent_both']
                elif cl:        sm = params['stop_mult_ent_caution']
                else:           sm = params['stop_mult_ent_normal']
                stop_loss = curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale']
        else:
            if prev['Close'] < stop_loss:
                in_position  = 0
                current_size = 0.0
                stop_loss    = np.nan
            else:
                sm        = params['stop_mult_pos_caution'] if curr['Caution_Long'] else params['stop_mult_pos_normal']
                stop_loss = max(stop_loss, curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale'])

        position[i]      = in_position
        position_size[i] = current_size
        stop_arr[i]      = stop_loss

    df['position']      = position
    df['position_size'] = position_size
    df['stop_loss']     = stop_arr

    # ── cleanup ───────────────────────────────────────────────────────────────
    indicator_cols = ['EMA', 'ATR_Cau', 'ADX_14', 'Swing_Hi_Cau', 'Vol_MA', 'OBV_MA']
    df['position']      = df['position'].fillna(0).astype(int)
    df['position_size'] = df['position_size'].fillna(0.0)
    df['stop_loss']     = df['stop_loss'].fillna(0.0)

    return df, indicator_cols

---
## Run Walk-Forward
Simulates how the strategy would have performed if re-optimised periodically
in live trading, and exposes whether good IS performance survives unseen data.

**Folds Setup**
| Parameter | Description | Guidance |
|---|---|---|
| `TRAIN_BARS` | Bars per training window | Aim for 2:1 to 3:1 ratio vs `TEST_BARS` |
| `TEST_BARS` | Bars per test window | `365` = ~1 year on daily data |
| `BURNIN_BARS` | Bars prepended to test for indicator warmup | Match your longest indicator period |
| `N_TRIALS` | Optuna trials per fold | 300–500 for daily; more = better but slower.10-20 trials per free parameter to avoid overfit |
| `COST` | Round-trip cost per trade | `0.001` = 0.1% |

Use `N_TRIALS` as robustness dia: if OOS degrades sharply as you increase it from 100→200→300, direct signal your parameter space still has too many degrees of freedom relative to the information content of the training window (consider decreasing). 

**Score and Rejection** — use to calibrate what Optuna optimises IS: default `score_fn(m)` uses weighted basket of Sharpe, Calmar and Return, normalised using their "max" value; default `reject_fn(m)` discards runs failing certain criteria that limits credibility.

> Pay attention to the **degradation ratio** — OOS/IS Sharpe reveals overfitting.

In [11]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 1050  
TEST_BARS   = 137   
BURNIN_BARS = 100   
N_TRIALS    = 400  
COST        = 0.001 

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
# Modify weights or swap components. Must return a float (higher = better).

def score_fn(m):
    SHARPE_MAX = 2.5
    CALMAR_MAX = 7
    RETURN_MAX = 14 #Calibrate to max of IS periods

    calmar = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0

    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)

    return 0.50 * s + 0.30 * c + 0.20 * r

# ── REJECTION CRITERIA ────────────────────────────────────────────────────────
# Trials that return True are discarded (score → -999).

def reject_fn(m):
    if m is None:                      return True
    if m['num_trades']    < 10:        return True
    if m['win_rate']      < 0.3:      return True
    if m['max_drawdown']  < -0.7:     return True
    if m['profit_factor'] < 0.6:       return True
    return False


results = walk_forward(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,
    score_fn     = score_fn,    # ← your notebook definition
    reject_fn    = reject_fn,   # ← your notebook definition
    save_csv     = None,
)

UPDATED WALK_FORWARD FILE IS RUNNING
Walk-forward: 7 fold(s)  train=1050  test=137  burnin=100  trials=400
  Fold 1: train 2020-09-22 → 2023-08-07  | test 2023-08-08 → 2023-12-22
  Fold 2: train 2021-02-06 → 2023-12-22  | test 2023-12-23 → 2024-05-07
  Fold 3: train 2021-06-23 → 2024-05-07  | test 2024-05-08 → 2024-09-21
  Fold 4: train 2021-11-07 → 2024-09-21  | test 2024-09-22 → 2025-02-05
  Fold 5: train 2022-03-24 → 2025-02-05  | test 2025-02-06 → 2025-06-22
  Fold 6: train 2022-08-08 → 2025-06-22  | test 2025-06-23 → 2025-11-06
  Fold 7: train 2022-12-23 → 2025-11-06  | test 2025-11-07 → 2026-03-23

Fixed (5): ['risk_per_trade', 'max_leverage', 'stop_mult_pos_caution', 'stop_atr_scale', 'vol_ma_period']
Free  (14): ['ema_span', 'swing_caution', 'swing_stop', 'atr_caution', 'atr_stop', 'atr_size', 'adx_override', 'stop_mult_pos_normal', 'stop_mult_ent_both', 'stop_mult_ent_caution', 'stop_mult_ent_short', 'stop_mult_ent_normal', 'obv_ma_period', 'obv_lookback']

───────────────────

  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 2.21  Return: 1210.95%  DD: -28.61%  Calmar: 5.05  Trades: 37
  OOS → Sharpe: 4.06  Return: 126.93%  DD: -14.62%  Calmar: 53.88  Trades: 4

  Best params: {'risk_per_trade': 0.0468, 'max_leverage': 2.5, 'stop_mult_pos_caution': 0.8423, 'stop_atr_scale': 1.9, 'vol_ma_period': 31, 'ema_span': 22, 'swing_caution': 5, 'swing_stop': 7, 'atr_caution': 20, 'atr_stop': 14, 'atr_size': 11, 'adx_override': 41, 'stop_mult_pos_normal': 1.419906390194898, 'stop_mult_ent_both': 2.0938384844435345, 'stop_mult_ent_caution': 0.7643541617899999, 'stop_mult_ent_short': 0.874758382456668, 'stop_mult_ent_normal': 0.9772890063025896, 'obv_ma_period': 13, 'obv_lookback': 15}

────────────────────────────────────────────────────────────
Fold 2/7  train: 2021-02-06 → 2023-12-22  test: 2023-12-23 → 2024-05-07


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 2.23  Return: 966.95%  DD: -18.99%  Calmar: 6.72  Trades: 38
  OOS → Sharpe: 0.89  Return: 9.75%  DD: -13.34%  Calmar: 2.11  Trades: 4

  Best params: {'risk_per_trade': 0.0468, 'max_leverage': 2.5, 'stop_mult_pos_caution': 0.8423, 'stop_atr_scale': 1.9, 'vol_ma_period': 31, 'ema_span': 20, 'swing_caution': 12, 'swing_stop': 5, 'atr_caution': 17, 'atr_stop': 23, 'atr_size': 5, 'adx_override': 42, 'stop_mult_pos_normal': 1.9525085791068644, 'stop_mult_ent_both': 2.021832863125541, 'stop_mult_ent_caution': 0.5067189355223436, 'stop_mult_ent_short': 0.16857424101886775, 'stop_mult_ent_normal': 0.9004507146388122, 'obv_ma_period': 22, 'obv_lookback': 19}

────────────────────────────────────────────────────────────
Fold 3/7  train: 2021-06-23 → 2024-05-07  test: 2024-05-08 → 2024-09-21


...


  IS  → Sharpe: 1.72  Return: 501.76%  DD: -23.63%  Calmar: 3.67  Trades: 32
  OOS → Sharpe: -2.52  Return: -19.51%  DD: -23.53%  Calmar: -1.87  Trades: 5

  Best params: {'risk_per_trade': 0.0468, 'max_leverage': 2.5, 'stop_mult_pos_caution': 0.8423, 'stop_atr_scale': 1.9, 'vol_ma_period': 31, 'ema_span': 20, 'swing_caution': 3, 'swing_stop': 6, 'atr_caution': 22, 'atr_stop': 16, 'atr_size': 14, 'adx_override': 43, 'stop_mult_pos_normal': 1.3544994191820237, 'stop_mult_ent_both': 0.6937291358731517, 'stop_mult_ent_caution': 0.8493903571688963, 'stop_mult_ent_short': 0.5634552363514166, 'stop_mult_ent_normal': 0.9557625711161569, 'obv_ma_period': 32, 'obv_lookback': 19}

────────────────────────────────────────────────────────────
Fold 6/7  train: 2022-08-08 → 2025-06-22  test: 2025-06-23 → 2025-11-06


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 1.60  Return: 413.78%  DD: -21.08%  Calmar: 3.64  Trades: 35
  OOS → Sharpe: 1.28  Return: 19.01%  DD: -16.28%  Calmar: 3.63  Trades: 5

  Best params: {'risk_per_trade': 0.0468, 'max_leverage': 2.5, 'stop_mult_pos_caution': 0.8423, 'stop_atr_scale': 1.9, 'vol_ma_period': 31, 'ema_span': 20, 'swing_caution': 10, 'swing_stop': 8, 'atr_caution': 26, 'atr_stop': 23, 'atr_size': 10, 'adx_override': 45, 'stop_mult_pos_normal': 1.8482989785868669, 'stop_mult_ent_both': 1.6378047689189699, 'stop_mult_ent_caution': 0.34364753161338446, 'stop_mult_ent_short': 1.1385837514553943, 'stop_mult_ent_normal': 1.4607185317327398, 'obv_ma_period': 15, 'obv_lookback': 13}

────────────────────────────────────────────────────────────
Fold 7/7  train: 2022-12-23 → 2025-11-06  test: 2025-11-07 → 2026-03-23


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 1.50  Return: 322.08%  DD: -21.98%  Calmar: 2.96  Trades: 33
  OOS → Sharpe: -0.20  Return: -1.72%  DD: -8.30%  Calmar: -0.54  Trades: 4

  Best params: {'risk_per_trade': 0.0468, 'max_leverage': 2.5, 'stop_mult_pos_caution': 0.8423, 'stop_atr_scale': 1.9, 'vol_ma_period': 31, 'ema_span': 20, 'swing_caution': 10, 'swing_stop': 4, 'atr_caution': 25, 'atr_stop': 23, 'atr_size': 14, 'adx_override': 42, 'stop_mult_pos_normal': 1.1653806191742895, 'stop_mult_ent_both': 1.5990362336458064, 'stop_mult_ent_caution': 0.8979679250163337, 'stop_mult_ent_short': 0.9206076118333862, 'stop_mult_ent_normal': 1.0490269118472875, 'obv_ma_period': 34, 'obv_lookback': 24}

════════════════════════════════════════════════════════════
WALK-FORWARD SUMMARY
════════════════════════════════════════════════════════════

Out-of-sample across 7 fold(s):
  Avg Sharpe:       0.70
  Avg Return:       21.2%
  Avg Max Drawdown: -15.3%
  Avg Calmar:       8.56
  Avg Trades/fold:  5
  Folds profitable:

---
## Granular Results and Parameter Stability

Per-fold IS vs OOS performance. Each row is one fold — compare `train_*` vs `test_*` columns to assess overfitting.

| Column | Description |
|---|---|
| `*_sharpe` `*_return` `*_drawdown` `*_calmar` | Core performance metrics |
| `*_trades` `*_winrate` `*_profit_factor` | Trade statistics |
| `optuna_score` | Best score achieved on training window |
| `param_*` | Best parameter values per fold e.g. `param_ema_span` |

**Concensus Parameters** - use to anchor: the engine determines stability using the coefficient of variation (CV) — the standard deviation of a parameter's best values across all folds divided by their median.

>CV < 0.15: indicates the strategy  relies on value rather than it being noise-fitted to a specific period — making it safe to fix for future runs. A high CV means the parameter is period-sensitive and should stay free.

In [12]:
# ── fold summary table ────────────────────────────────────────────────────────
display_cols = [
    'train_start', 'train_end',
    'test_start', 'test_end',
    'train_sharpe', 'test_sharpe',
    'train_return', 'test_return',
    'train_drawdown', 'test_drawdown',
    'test_trades',  'test_winrate', 'optuna_score',
]
display(results['results_df'][display_cols].round(2))

# ── parameter stability ───────────────────────────────────────────────────────
stability = results['stability_df'].copy()
stability['stable'] = stability['stable'].map({True: '✓', False: ''})
stability['fixed']  = stability['fixed'].map({True: '✓', False: ''})
stability = stability[['param', 'median', 'std', 'cv', 'stable', 'fixed']].round(2)
display(stability.sort_values('cv'))

# ── consensus params ──────────────────────────────────────────────────────────
stable = results['stability_df'][results['stability_df']['cv'] < 0.15]

print('Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:')
for _, row in stable.iterrows():
    v = results['consensus_params'][row['param']]
    v_fmt = int(round(v)) if isinstance(v, float) and v == int(v) else round(v, 4) if isinstance(v, float) else v
    print(f"    '{row['param']}': {v_fmt},")
    
print('\nConsensus parameters (median across folds):')
for k, v in results['consensus_params'].items():
    print(f'  {k:<30} = {round(v, 2) if isinstance(v, float) else v}')

,train_start,train_end,test_start,test_end,train_sharpe,test_sharpe,train_return,test_return,train_drawdown,test_drawdown,test_trades,test_winrate,optuna_score
0,2020-09-22,2023-08-07,2023-08-08,2023-12-22,2.21,4.06,12.11,1.27,-0.29,-0.15,4,0.50,0.92
1,2021-02-06,2023-12-22,2023-12-23,2024-05-07,2.23,0.89,9.67,0.10,-0.19,-0.13,4,0.50,0.88
2,2021-06-23,2024-05-07,2024-05-08,2024-09-21,2.31,0.99,14.58,0.10,-0.29,-0.14,4,0.25,0.96
3,2021-11-07,2024-09-21,2024-09-22,2025-02-05,1.84,0.42,4.97,0.03,-0.14,-0.17,6,0.33,0.74
4,2022-03-24,2025-02-05,2025-02-06,2025-06-22,1.72,-2.52,5.02,-0.20,-0.24,-0.24,5,0.20,0.72
5,2022-08-08,2025-06-22,2025-06-23,2025-11-06,1.60,1.28,4.14,0.19,-0.21,-0.16,5,0.40,0.68
6,2022-12-23,2025-11-06,2025-11-07,2026-03-23,1.50,-0.20,3.22,-0.02,-0.22,-0.08,4,0.50,0.65


,param,median,std,cv,stable,fixed
9,max_leverage,2.50,0.00,0.00,✓,✓
16,vol_ma_period,31.00,0.00,0.00,✓,✓
10,stop_mult_pos_caution,0.84,0.00,0.00,✓,✓
8,risk_per_trade,0.05,0.00,0.00,✓,✓
7,stop_atr_scale,1.90,0.00,0.00,✓,✓
0,ema_span,20.00,0.70,0.03,✓,
3,atr_caution,24.00,3.92,0.16,,
11,stop_mult_pos_normal,1.70,0.30,0.17,,
15,stop_mult_ent_normal,0.98,0.17,0.18,,
4,atr_stop,23.00,5.12,0.22,,


Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:
    'ema_span': 20,
    'stop_atr_scale': 1.9,
    'risk_per_trade': 0.0468,
    'max_leverage': 2.5,
    'stop_mult_pos_caution': 0.8423,
    'vol_ma_period': 31,

Consensus parameters (median across folds):
  ema_span                       = 20
  swing_caution                  = 6
  swing_stop                     = 6
  atr_caution                    = 24
  atr_stop                       = 23
  atr_size                       = 10
  adx_override                   = 42
  stop_atr_scale                 = 1.9
  risk_per_trade                 = 0.05
  max_leverage                   = 2.5
  stop_mult_pos_caution          = 0.84
  stop_mult_pos_normal           = 1.7
  stop_mult_ent_both             = 1.64
  stop_mult_ent_caution          = 0.58
  stop_mult_ent_short            = 0.76
  stop_mult_ent_normal           = 0.98
  vol_ma_period                  = 31
  obv_ma_period                  = 32
  obv_lookback                   = 19


---
## Parameter Robustness Checks

### Plateau Analysis
Sweep each free parameter across its range while holding others at consensus (median) value then evaluates the `score` at each point by backtesting over entire lookback.

The stability table (CV across folds) tells you *"does the optimizer consistently pick the same value?"*  

Plateau analysis tells you *"if that value were slightly wrong, would performance collapse?"*  

**Plateau %** - what fraction of each parameter's range stays within `threshold`% (default 20) of peak score: >60% = `robust plateau`, 30–60% = `moderate`, <30% = `fragile` (consider anchoring). `N/A` means every sweep point failed rejection filters — the strategy is completely intolerant of movement on that dimension.

>Run time: `n_free_params` × `n_steps`

### Perturbation test
Jitters all free parameters by ±5/10/20% of their range (50 random samples per offset range). Measures how much the score degrades vs the base

Test whether optimum is a broad hill in `#free params`-D space or a narrow spike

**>15%:** fragile optimum, consider reducing free parameters

In [13]:
# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
sweep_results = plateau_analysis(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    n_steps      = 20, #Adjust for number of steps around concensus per parameter
)

# ── text verdicts ──────────────────────────────────────────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params = results['consensus_params'],
    stability_df = results['stability_df'],  
    threshold   = 0.15, #Adjust for % around peak score
)


# ── neighbouahood perturbation ────────────────────────────────────────────
# Randomly jitters ALL free params simultaneously.
# If mean score degrades >15% at ±10% offset, the optimum is fragile.

perturb_df = perturbation_test(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    pct_offsets  = (0.05, 0.10, 0.20),   # ±5%, ±10%, ±20% of range
    n_samples    = 50,                     # random perturbations per offset level
)


═══════════════════════════════════════════════════════════════════════════════════════════
PLATEAU ANALYSIS — PARAMETER ROBUSTNESS
═══════════════════════════════════════════════════════════════════════════════════════════
Parameter                 Consensus Peak Score  Plateau %  Fold CV Verdict                 
──────────────────────── ────────── ────────── ────────── ──────── ────────────────────────
atr_caution                      24      0.837      100.0%    0.163 Robust                  
obv_ma_period                    32      0.832      100.0%    0.299 Robust                  
obv_lookback                     19      0.830      100.0%    0.244 Robust                  
atr_size                         10      0.830      100.0%    0.365 Robust                  
swing_caution                     6      0.827      100.0%    0.503 Robust                  
stop_mult_ent_caution        0.5790      0.815      100.0%    0.361 Robust                  
stop_mult_ent_short          0.75

### 1-D sweep charts:
| Element | Meaning | Good | Bad |
|---------|---------|------|-----|
| **Blue curve** | Composite score at each value of the parameter, with all others held at consensus | Flat-topped curve — performance is insensitive to the exact value | Narrow spike — optimizer latched onto one specific value, everything nearby is worse |
| **Red dashed line** | Where the consensus value sits | On the flat top of the curve | On a steep slope or at the edge of a cliff |
| **Green dashed line** | Cutoff at 80% of peak score — the boundary between plateau and non-plateau | Blue curve stays above this line across most of the range | Blue curve dips below it quickly either side of the peak |
| **Green shading** | Plateau region — all values where the score stays within 20% of the peak | Wide green band spanning most of the range (robust) | Thin sliver or no shading at all (fragile/overfit) |


 If concensus on steep slope: parameter **REGIME SENSITIVE** - do not fix, backtests are disagreeing, want to fix parameters on flat top.

In [14]:
# ── visual sweep curves ───────────────────────────────────────────────────
plot_plateau_analysis(
    sweep_results    = sweep_results,
    consensus_params = results['consensus_params'],
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    threshold        = 0.20,
    show             = False,
    save_html        = None,
)

---
## Results Charts and Cost Stress Test

| Parameter | Description | Default |
|---|---|---|
| `show_fold_perf` | IS vs OOS bars for return, Sharpe, drawdown per fold | `False` |
| `show_param_evol` | Parameter evolution across folds with ±1 std bands | `False` |
| `show_oos_equity` | Combined OOS equity curve + drawdown with fold boundaries | `True` |
| `show_trades` | Overlay entry/exit markers on OOS equity chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | `None` |
| `save_html_dir` | Directory path to save charts as HTML files, or `None` | `None` |

**Cost Stress Test:** re-run the combined OOS backtest at 1×, 1.5×, 2×, 3× the base cost. Fragile strategies collapse; robust ones degrade gradually.

In [15]:
plot_walk_forward_results(
    results          = results,
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    benchmark_data   = df,
    show             = True,
    save_html_dir    = None,
    show_fold_perf   = False,   # IS vs OOS bars by fold
    show_param_evol  = False,   # parameter evolution across folds
    show_oos_equity  = True,   # combined OOS equity curve
    show_trades      = False,  # trade markers on OOS equity chart
)

# ── transaction cost stress test ──────────────────────────────────────────

if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test')


═══════════════════════════════════════════════════════════════════════════
TRANSACTION COST STRESS TEST
═══════════════════════════════════════════════════════════════════════════
    Cost   Mult   Sharpe     Return      MaxDD   Calmar       PF
──────── ────── ──────── ────────── ────────── ──────── ────────
  0.0010   1.0x     0.73     79.06%    -49.90%     0.50     1.57
  0.0015   1.5x     0.71     73.41%    -49.95%     0.47     1.57
  0.0020   2.0x     0.68     67.94%    -50.00%     0.44     1.57
  0.0030   3.0x     0.63     57.50%    -50.10%     0.38     1.57


In [16]:
results['oos_combined_df'].to_pickle('avax_oos.pkl')